# 事前学習モデルを用いた転移学習・ファインチューニング

---
## 目的
ImageNetで学習済みのモデルをCIFAR100に適用し，「特徴抽出（feature extraction）」と「ファインチューニング（fine-tuning）」という2つの転移学習の手法の違いを理解する．`torchvision_models.ipynb`で読み込んだResNet50による特徴抽出と，`timm_models.ipynb`で読み込んだEfficientNetによるファインチューニングを，それぞれ実践する．

## モジュールのインポート
`timm`は標準ではインストールされていないため，はじめに`pip install`でインストールしたのち，必要なモジュールをインポートし，GPUを使用した計算が可能かどうかを確認する．

In [ ]:
!pip install -q timm

from time import time

import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torchvision.models import resnet50, ResNet50_Weights
import timm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

## データセットの読み込みと確認
CIFAR100データセットを読み込みます．事前学習済みモデルはImageNet（224×224，`mean=[0.485, 0.456, 0.406]`, `std=[0.229, 0.224, 0.225]`で正規化）で学習されているため，CIFAR100の32×32画像もモデルの入力サイズ（224×224）にリサイズし，同じ統計量で正規化する必要があります．学習時と異なる前処理を行うと，事前学習済みの特徴抽出能力を十分に活かせません．

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

transform_train = transforms.Compose([transforms.Resize(224),
                                       transforms.RandomHorizontalFlip(),
                                       transforms.ToTensor(),
                                       transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)])
transform_test = transforms.Compose([transforms.Resize(224),
                                      transforms.ToTensor(),
                                      transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)])

train_data = torchvision.datasets.CIFAR100(root="./", train=True, transform=transform_train, download=True)
test_data = torchvision.datasets.CIFAR100(root="./", train=False, transform=transform_test, download=True)

print(len(train_data), len(test_data))

## 転移学習の2つのアプローチ
事前学習済みモデルを新しいデータセットに適用する方法には，大きく分けて次の2つがあります．

- **特徴抽出（Feature Extraction）**：事前学習済みのbackbone（畳み込み部分）のパラメータを凍結（`requires_grad=False`）し，新しいデータセットのクラス数に合わせて置き換えた最終層（分類ヘッド）のみを学習します．ImageNetで学習した汎用的な特徴抽出能力をそのまま利用するため，少ないデータ・短い学習時間でも安定して学習できます．
- **ファインチューニング（Fine-tuning）**：backboneも含めて全てのパラメータを学習対象とします．事前学習済みの重みを大きく崩さないよう，通常はゼロから学習する場合よりも小さい学習率を用います．データ量が十分にある場合，特徴抽出よりも高い精度が期待できます．

以下では，torchvisionで読み込んだResNet50を用いて特徴抽出を，timmで読み込んだEfficientNet-B0を用いてファインチューニングを行い，両者を比較します．

## torchvision: ResNet50による特徴抽出
`weights=ResNet50_Weights.IMAGENET1K_V2`で事前学習済みのResNet50を読み込み，全パラメータを凍結したのち，最終の全結合層（`model.fc`）のみをCIFAR100のクラス数（100）に置き換えます．新しく作成した層は，デフォルトで`requires_grad=True`のまま初期化されます．

In [ ]:
model_fe = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)

# backboneの全パラメータを凍結（勾配を計算しない）
for param in model_fe.parameters():
    param.requires_grad = False

# 最終の全結合層をCIFAR100のクラス数(100)に置き換える
model_fe.fc = nn.Linear(model_fe.fc.in_features, 100)
model_fe = model_fe.to(device)

# 学習対象となるパラメータ数を確認
n_trainable = sum(p.numel() for p in model_fe.parameters() if p.requires_grad)
n_total = sum(p.numel() for p in model_fe.parameters())
print(f'学習対象のパラメータ数: {n_trainable} / {n_total}')

optimizer_fe = torch.optim.SGD(model_fe.fc.parameters(), lr=0.01, momentum=0.9, weight_decay=5e-4)

## 学習（特徴抽出）
学習対象は新しく追加した全結合層のみであるため，比較的少ないエポック数でも学習が進みます．ここではミニバッチサイズを64，学習エポック数を5とします．

In [ ]:
batch_size = 64
epoch_num = 5

train_loader = torch.utils.data.DataLoader(train_data, batch_size=batch_size, shuffle=True, num_workers=2)

criterion = nn.CrossEntropyLoss()
criterion = criterion.to(device)

model_fe.train()

train_start = time()
for epoch in range(1, epoch_num + 1):
    sum_loss = 0.0
    count = 0

    for image, label in train_loader:
        image = image.to(device)
        label = label.to(device)

        y = model_fe(image)

        loss = criterion(y, label)
        model_fe.zero_grad()
        loss.backward()
        optimizer_fe.step()

        sum_loss += loss.item()
        pred = torch.argmax(y, dim=1)
        count += torch.sum(pred == label)

    print("epoch: {}, mean loss: {}, mean accuracy: {}, elapsed time: {}".format(
        epoch, sum_loss / len(train_loader), count.item() / len(train_data), time() - train_start))

## テスト（特徴抽出）
学習した特徴抽出モデルを用いて評価を行います．

In [ ]:
test_loader = torch.utils.data.DataLoader(test_data, batch_size=100, shuffle=False)

model_fe.eval()

count = 0
with torch.no_grad():
    for image, label in test_loader:
        image = image.to(device)
        label = label.to(device)

        y = model_fe(image)

        pred = torch.argmax(y, dim=1)
        count += torch.sum(pred == label)

print("test accuracy (feature extraction): {}".format(count.item() / len(test_data)))

## timm: EfficientNetによるファインチューニング
timmでは，`timm.create_model(model_name, pretrained=True, num_classes=100)`のように`num_classes`を指定するだけで，分類ヘッドが指定したクラス数用に自動的に置き換えられた状態でモデルを取得できます．ここでは，backboneを凍結せず全パラメータを学習対象とし，事前学習済みの重みを大きく崩さないよう学習率を小さめ（0.001）に設定してファインチューニングを行います．

In [ ]:
model_ft = timm.create_model('efficientnet_b0', pretrained=True, num_classes=100)
model_ft = model_ft.to(device)

# 全パラメータが学習対象になっていることを確認
n_trainable = sum(p.numel() for p in model_ft.parameters() if p.requires_grad)
n_total = sum(p.numel() for p in model_ft.parameters())
print(f'学習対象のパラメータ数: {n_trainable} / {n_total}')

# ファインチューニングでは，特徴抽出のときより小さい学習率を用いる
optimizer_ft = torch.optim.SGD(model_ft.parameters(), lr=0.001, momentum=0.9, weight_decay=5e-4)

## 学習（ファインチューニング）
backboneも含めて全パラメータを更新するため，特徴抽出よりも1エポックあたりの計算コストが大きくなります．エポック数は特徴抽出と同じく5とします．

In [ ]:
epoch_num = 5

model_ft.train()

train_start = time()
for epoch in range(1, epoch_num + 1):
    sum_loss = 0.0
    count = 0

    for image, label in train_loader:
        image = image.to(device)
        label = label.to(device)

        y = model_ft(image)

        loss = criterion(y, label)
        model_ft.zero_grad()
        loss.backward()
        optimizer_ft.step()

        sum_loss += loss.item()
        pred = torch.argmax(y, dim=1)
        count += torch.sum(pred == label)

    print("epoch: {}, mean loss: {}, mean accuracy: {}, elapsed time: {}".format(
        epoch, sum_loss / len(train_loader), count.item() / len(train_data), time() - train_start))

## テスト（ファインチューニング）
学習したファインチューニングモデルを用いて評価を行います．

In [ ]:
model_ft.eval()

count = 0
with torch.no_grad():
    for image, label in test_loader:
        image = image.to(device)
        label = label.to(device)

        y = model_ft(image)

        pred = torch.argmax(y, dim=1)
        count += torch.sum(pred == label)

print("test accuracy (fine-tuning): {}".format(count.item() / len(test_data)))

## まとめ：特徴抽出とファインチューニングの比較

| | 特徴抽出（ResNet50） | ファインチューニング（EfficientNet-B0） |
|---|---|---|
| 学習対象パラメータ | 新しく追加した全結合層のみ | 全パラメータ |
| 1エポックあたりの計算コスト | 小さい（backboneのbackwardが不要） | 大きい（backboneも含めて逆伝播） |
| 学習率の目安 | 通常のスクラッチ学習と同程度 | 小さめ（事前学習済み重みを崩さないように） |
| 必要なデータ量・向いている場面 | 少量のデータでも安定 | データ量が多い場合に高精度が期待できる |

どちらの手法が適しているかは，用意できるデータ量・計算資源・目標精度によって異なります．一般には，データ量が少ない場合や計算資源が限られる場合は特徴抽出，データ量が多く十分な学習時間が確保できる場合はファインチューニングが選ばれます．

## 課題

1. ResNet50を（backboneを凍結せず）ファインチューニングした場合と，本ノートブックの特徴抽出の結果とでテスト精度がどのように変わるか比較しましょう．

2. EfficientNet-B0を特徴抽出（backboneを凍結）で学習した場合と，本ノートブックのファインチューニングの結果とを比較しましょう．

3. backboneの一部の層のみを凍結解除する「部分的なファインチューニング」（例えば，ResNet50の`layer4`と`fc`のみを学習対象にする）を試し，全パラメータを学習する場合や特徴抽出の場合と比較しましょう．